In [1]:
import pdfplumber
import re
import pandas as pd

INCOME_STATEMENT_TITLES = [
    "Statement of Profit or Loss and Other Comprehensive Income",
    "Consolidated Statement of Income",
    "Consolidated Statement of Earnings",
    "Consolidated Statement of Operations",
]

BALANCE_SHEET_TITLES = [
    "Statement of Financial Position",
    "Consolidated Balance Sheet",
    "Consolidated Financial Position",
    "Consolidated Statement of Assets, Liabilities and Equity",
]

INCOME_STATEMENT_ITEMS = {
    "Revenue": ["Revenue", "Sales", "Turnover"],
    "Cost of goods sold": ["Cost of goods sold", "Cost of sales", "Cost of revenue", "Materials, transportation and storage"],
    "Gross Profit/(Loss)": ["Gross profit", "Gross loss", "Gross profit/(loss)", "Gross profit or (loss)"],
    "Operating expenses": ["Operating expenses", "Operating costs", "Operating expenditure"],
    "Operating Result": ["Operating result", "Operating profit", "Operating loss", "Operating profit or (loss)", "Operating result"],
    "Financial/Interest income": ["Financial income", "Finance income", "Interest income"],
    "Financial/Interest expenses": ["Financial expenses", "Finance costs", "Interest expense", "Interest expenses"],
    "Depreciation": ["Depreciation", "Depreciation and amortisation", "Depreciation (right-of-use assets)", "Depreciation and amortisation (PP&E and intangible assets)"],
    "Profit before income tax": ["Profit before income tax", "Profit before tax", "Profit before taxation"],
    "Income tax income/(expenses)": ["Income tax", "Income tax expense", "Income tax (expense)", "Income tax (income)", "Income tax credit"],
    "Net Result": ["Net result", "Net profit", "Net income", "Profit for the year", "Profit attributable"],
}

BALANCE_SHEET_ITEMS = {
    "Cash and cash equivalents": ["Cash and cash equivalents", "Cash & cash equivalents"],
    "Due from related parties": ["Due from related parties", "Amounts due from related parties", "Related party receivables"],
    "Trade and other receivables": ["Trade and other receivables", "Trade receivables", "Receivables"],
    "Inventories": ["Inventories"],
    "Total Current Assets": ["Total current assets"],
    "Property, plant and equipment": ["Property, plant and equipment", "Property plant and equipment", "PP&E"],
    "Right-of-use assets": ["Right-of-use assets", "ROU assets"],
    "Total Non-Current Assets": ["Total non-current assets", "Total non current assets"],
    "Total Assets": ["Total assets"],
    "Short-term borrowings": ["Short-term borrowings", "Short term borrowings", "Current borrowings"],
    "Short-term Lease Liabilities": ["Lease liabilities (current)", "Current lease liabilities", "Lease liabilities - current"],
    "Trade and other payables": ["Trade and other payables", "Trade payables", "Payables"],
    "Total Current Liabilities": ["Total current liabilities"],
    "Long-term borrowings": ["Long-term borrowings", "Long term borrowings", "Non-current borrowings"],
    "Long-term Lease liabilities": ["Lease liabilities (non-current)", "Non-current lease liabilities", "Lease liabilities - non-current"],
    "Total Non-Current Liabilities": ["Total non-current liabilities", "Total non current liabilities"],
    "Total Liabilities": ["Total liabilities"],
    "Issued capital": ["Issued capital", "Share capital"],
    "Retained earnings/(Accumulated losses)": ["Retained earnings", "Accumulated losses", "Accumulated loss"],
    "Total Equity": ["Total equity", "Equity attributable"],
}

def detect_years_from_statement(text, max_years=5):
    """Extract years from statement text using regex pattern."""
    year_pattern = r'\b(19\d{2}|20\d{2})\b'
    header_text = text[:2000] if len(text) > 2000 else text
    years = re.findall(year_pattern, header_text)
    if len(years) == 0:
        years = re.findall(year_pattern, text)
    unique_years = sorted(set(int(y) for y in years), reverse=True)
    detected = unique_years[:max_years]
    return detected if detected else [2025, 2024]

def find_statement_page(pdf_path, titles_list, start_page=10, max_pages=100):
    """Find page containing statement by matching priority-ordered titles."""
    with pdfplumber.open(pdf_path) as pdf:
        end_page = min(start_page + max_pages, len(pdf.pages))
        for page_idx in range(start_page, end_page):
            try:
                page_text = pdf.pages[page_idx].extract_text()
                if page_text is None:
                    continue
                page_header = page_text[:300].lower()
                for title_part in titles_list:
                    if title_part.lower() in page_header:
                        lines = page_text.split('\n')
                        has_numbers = False
                        for line in lines[5:20]:
                            if re.search(r'\d{3,}', line):
                                has_numbers = True
                                break
                        if has_numbers:
                            print(f"✅ Found {title_part} on page {page_idx + 1}")
                            return page_idx, page_text
            except Exception:
                continue
    return None, None

def extract_metric_value(line, num_years):
    """Extract numbers from a line for specified number of years."""
    numbers = [float(n.replace(',', '')) for n in re.findall(r'[\d,]+\.?\d*', line)
               if n and n.replace(',', '').replace('.', '').isdigit()]
    if len(numbers) >= num_years:
        return numbers[-num_years:]
    return None

def find_matching_line(statement_text, keywords, years_count):
    """Find a line in statement that matches any of the keywords and has numbers."""
    lines = statement_text.split('\n')
    for line in lines:
        line_lower = line.lower()
        for keyword in keywords:
            if keyword.lower() in line_lower:
                values = extract_metric_value(line, years_count)
                if values:
                    return line[:60].strip(), values
    return None, None

def extract_financial_data(pdf_path):
    """Extract financial data from PDF."""
    print("\n🔍 STEP 1: Finding statements...\n")

    income_page_idx, income_page_text = find_statement_page(pdf_path, INCOME_STATEMENT_TITLES)
    balance_page_idx, balance_page_text = find_statement_page(pdf_path, BALANCE_SHEET_TITLES)

    if income_page_text is None:
        raise ValueError("❌ Could not find Income Statement")
    if balance_page_text is None:
        raise ValueError("❌ Could not find Balance Sheet")

    print("\n📅 STEP 2: Detecting years...\n")
    income_years = detect_years_from_statement(income_page_text)
    balance_years = detect_years_from_statement(balance_page_text)

    print(f"Income Statement: {income_years}")
    print(f"Balance Sheet: {balance_years}")

    print("\n📊 STEP 3: Extracting specified metrics...\n")


    financial_data = {
        'income_statement': {},
        'balance_sheet': {},
        'metadata': {
            'income_page': income_page_idx + 1,
            'balance_page': balance_page_idx + 1,
            'income_years': income_years,
            'balance_years': balance_years
        }
    }
    
 
    for item_name in INCOME_STATEMENT_ITEMS.keys():
        financial_data['income_statement'][item_name] = dict(zip(income_years, [0.0] * len(income_years)))
    
 
    for item_name in BALANCE_SHEET_ITEMS.keys():
        financial_data['balance_sheet'][item_name] = dict(zip(balance_years, [0.0] * len(balance_years)))

    print("Income Statement metrics:")
    for item_name, aliases in INCOME_STATEMENT_ITEMS.items():
        metric_name, values = find_matching_line(income_page_text, aliases, len(income_years))
        if metric_name and values:
            financial_data['income_statement'][item_name] = dict(zip(income_years, values))
            print(f"  ✓ {item_name}")
        else:
            print(f"  ✗ {item_name} (not found - set to 0)")

    print("\nBalance Sheet metrics:")
    for item_name, aliases in BALANCE_SHEET_ITEMS.items():
        metric_name, values = find_matching_line(balance_page_text, aliases, len(balance_years))
        if metric_name and values:
            financial_data['balance_sheet'][item_name] = dict(zip(balance_years, values))
            print(f"  ✓ {item_name}")
        else:
            print(f"  ✗ {item_name} (not found - set to 0)")

    print(f"\n✅ Extracted {sum(1 for item in financial_data['income_statement'].values() if any(v != 0 for v in item.values()))}/{len(INCOME_STATEMENT_ITEMS)} income metrics")
    print(f"✅ Extracted {sum(1 for item in financial_data['balance_sheet'].values() if any(v != 0 for v in item.values()))}/{len(BALANCE_SHEET_ITEMS)} balance sheet metrics")


    print("\n🧮 STEP 4: Calculating derived metrics...\n")
    
    # INCOME STATEMENT CALCULATIONS
    
    # Calculate Gross Profit = Revenue - Cost of Goods Sold
    revenue_data = financial_data['income_statement'].get('Revenue', {})
    cogs_data = financial_data['income_statement'].get('Cost of goods sold', {})
    
    gross_profit = {}
    for year in income_years:
        revenue_val = revenue_data.get(year, 0.0)
        cogs_val = cogs_data.get(year, 0.0)
        gross_profit[year] = revenue_val - cogs_val
    financial_data['income_statement']['Gross Profit/(Loss)'] = gross_profit
    print(f"  ✓ Calculated Gross Profit/(Loss)")

    # BALANCE SHEET CALCULATIONS
    
    # Calculate Total Liabilities = Total Current Liabilities + Total Non-Current Liabilities
    current_liab = financial_data['balance_sheet'].get('Total Current Liabilities', {})
    non_current_liab = financial_data['balance_sheet'].get('Total Non-Current Liabilities', {})
    
    total_liabilities = {}
    for year in balance_years:
        current_val = current_liab.get(year, 0.0)
        non_current_val = non_current_liab.get(year, 0.0)
        total_liabilities[year] = current_val + non_current_val
    financial_data['balance_sheet']['Total Liabilities'] = total_liabilities
    print(f"  ✓ Calculated Total Liabilities")

    return financial_data

# Execute extraction
pdf_path = r'C:\Users\Lim Chao Guan\OneDrive - Union International Trading Pte Ltd\Desktop\Pdf_project\2025_trafigura_annual_report.pdf'

try:
    financial_data = extract_financial_data(pdf_path)
    print("\n✅ Extraction complete!")
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()
    financial_data = None




🔍 STEP 1: Finding statements...

✅ Found Consolidated Statement of Income on page 42
✅ Found Statement of Financial Position on page 44

📅 STEP 2: Detecting years...

Income Statement: [2025, 2024]
Balance Sheet: [2025, 2024]

📊 STEP 3: Extracting specified metrics...

Income Statement metrics:
  ✓ Revenue
  ✓ Cost of goods sold
  ✗ Gross Profit/(Loss) (not found - set to 0)
  ✗ Operating expenses (not found - set to 0)
  ✓ Operating Result
  ✓ Financial/Interest income
  ✗ Financial/Interest expenses (not found - set to 0)
  ✓ Depreciation
  ✓ Profit before income tax
  ✓ Income tax income/(expenses)
  ✓ Net Result

Balance Sheet metrics:
  ✓ Cash and cash equivalents
  ✗ Due from related parties (not found - set to 0)
  ✓ Trade and other receivables
  ✓ Inventories
  ✓ Total Current Assets
  ✓ Property, plant and equipment
  ✓ Right-of-use assets
  ✓ Total Non-Current Assets
  ✓ Total Assets
  ✗ Short-term borrowings (not found - set to 0)
  ✗ Short-term Lease Liabilities (not found

In [2]:
import pandas as pd

income_df = pd.DataFrame(financial_data['income_statement']).T
income_df.index.name = 'Metric'
income_df.columns = ['2025', '2024']

balance_df = pd.DataFrame(financial_data['balance_sheet']).T
balance_df.index.name = 'Metric'
balance_df.columns = ['2025', '2024']

def format_financial_statement(df, title):
    """Format DataFrame to look like a professional financial statement"""
    
    bold_rows = [
        'Gross Profit/(Loss)', 'Operating Result', 'Profit before income tax', 'Net Result',
        'Total Current Assets', 'Total Non-Current Assets', 'Total Assets',
        'Total Current Liabilities', 'Total Non-Current Liabilities', 'Total Liabilities', 'Total Equity'
    ]
    
    print(f"\n{'=' * 80}")
    print(f"{title.upper()}")
    print(f"{'=' * 80}")
    print(f"{'Line Item':<50} {'2025':>12} {'2024':>12}")
    print(f"{'-' * 80}")
    
    for idx, row in df.iterrows():
      
        val_2025 = f"{row['2025']:,.2f}" if row['2025'] != 0 else "-"
        val_2024 = f"{row['2024']:,.2f}" if row['2024'] != 0 else "-"
        
        if idx == 'Gross Profit/(Loss)':
            print(f"{'·' * 80}")
        
        if idx in bold_rows:
            print(f"{idx:<50} {val_2025:>12} {val_2024:>12}")
            if idx in ['Net Result', 'Total Assets', 'Total Equity']:
                print(f"{'-' * 80}")
        else:
            print(f"  {idx:<48} {val_2025:>12} {val_2024:>12}")
    
    print(f"{'=' * 80}\n")

format_financial_statement(income_df, "Income Statement (in USD millions)")
format_financial_statement(balance_df, "Balance Sheet (in USD millions)")



INCOME STATEMENT (IN USD MILLIONS)
Line Item                                                  2025         2024
--------------------------------------------------------------------------------
  Revenue                                            240,268.00   243,201.80
  Cost of goods sold                                 228,238.90   231,355.80
················································································
Gross Profit/(Loss)                                   12,029.10    11,846.00
  Operating expenses                                          -            -
Operating Result                                       8,003.60     8,162.60
  Financial/Interest income                            1,478.60     1,768.90
  Financial/Interest expenses                                 -            -
  Depreciation                                         8,003.60     8,162.60
Profit before income tax                               3,011.70     2,838.30
  Income tax income/(expenses)  

In [3]:
import json
import requests
import time
import threading
from tqdm import tqdm

OLLAMA_URL = "http://localhost:11434/api/generate"

payload = {
    "income_statement": income_df.to_dict(orient="index"),
    "balance_sheet": balance_df.to_dict(orient="index"),
}

prompt = (
    "Analyze the following financial figures and provide structured output with these sections:\n\n"
    "【执行摘要 (Executive Summary)】\n"
    "Summarize key trends and year-over-year changes.\n\n"
    "【红旗警告 (Red Flags)】\n"
    "Identify any potential concerns or risks.\n\n"
    "【前景预测 (Forecast)】\n"
    "Provide growth outlook and future implications.\n\n"
    "【关键洞察 (Key Insights)】\n"
    "List 3-5 bullet points highlighting important findings.\n\n"
    f"Data (JSON): {json.dumps(payload, indent=2)}"
)

print("🤖 Mistral is thinking....\n")

try:
    start_time = time.time()
    response_received = threading.Event()
    
    
    pbar = tqdm(total=300, desc="Processing", unit="s", bar_format="{desc}: {bar} {elapsed}s elapsed")
    
    def update_progress():
        """Update progress bar every second until response is received"""
        while not response_received.is_set():
            elapsed = int(time.time() - start_time)
            pbar.n = min(elapsed, 300)
            pbar.refresh()
            time.sleep(1)
    
    progress_thread = threading.Thread(target=update_progress, daemon=True)
    progress_thread.start()
    
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": "mistral",
            "prompt": prompt,
            "stream": False,
            "temperature": 0  # Deterministic output for consistent financial analysis
        },
        timeout=300
    )
    
    response_received.set()
    progress_thread.join(timeout=2)
    pbar.close()
    
    if response.status_code == 200:
        result = response.json()
        elapsed = time.time() - start_time
        print(f"\n✅ Analysis complete in {elapsed:.1f}s\n")
        print(result['response'])
    else:
        print(f"Error {response.status_code}: {response.text}")
        
except requests.exceptions.ConnectionError:
    print("Cannot connect to Ollama. Make sure Ollama is running: ollama serve")
except requests.exceptions.Timeout:
    print("Request timed out.")
except Exception as e:
    print(f"Error: {e}")

🤖 Mistral is thinking....



Processing: █████▌     02:48s elapsed


✅ Analysis complete in 168.3s

 **Executive Summary**
- The company's revenue saw a minor decrease from $243,201.8 in 2024 to $240,268 in 2025, but this was offset by a significant improvement in gross profit, which increased from $11,846 in 2024 to $12,029.10 in 2025.
- The operating result also showed an increase, rising from $8,162.6 in 2024 to $8,003.6 in 2025. However, financial income decreased slightly from $1,768.9 in 2024 to $1,478.6 in 2025.
- The net result for the year 2025 was $2,665.5, a small decrease compared to $2,758.7 in 2024.

**Red Flags**
- Despite the increase in gross profit and operating result, the revenue decline could indicate potential challenges in business operations or market conditions.
- The significant decrease in financial income may raise concerns about the company's investment strategies or interest expenses.
- The increase in current liabilities from $47,146.9 in 2024 to $50,303.8 in 2025 might suggest a higher reliance on short-term borrowing or